# RAG Pipeline with MiniMax via OpenAI-Compatible API

In this notebook, you'll learn how to use **[MiniMax](https://www.minimax.io/)** large language models with **Haystack** through the OpenAI-compatible API.

MiniMax provides models like **MiniMax-M2.7** (204K context window) and **MiniMax-M2.7-highspeed**, which are accessible via an OpenAI-compatible endpoint. This means you can use Haystack's built-in `OpenAIChatGenerator` to work with MiniMax models — no extra integration packages required.

We'll build:
1. A **basic chat** example to verify the connection
2. A full **RAG pipeline** using MiniMax as the generator

## Setup

Install the required packages:

In [ ]:
!pip install -q -U haystack-ai datasets sentence-transformers

Set your MiniMax API key. You can get one from the [MiniMax Platform](https://platform.minimax.io/).

In [ ]:
import os
from getpass import getpass

if "MINIMAX_API_KEY" not in os.environ:
    os.environ["MINIMAX_API_KEY"] = getpass("Enter your MiniMax API key: ")

## Basic Chat with MiniMax

Since MiniMax provides an OpenAI-compatible API at `https://api.minimax.io/v1`, you can use
Haystack's `OpenAIChatGenerator` directly by setting `api_base_url` and passing the API key.

In [ ]:
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage
from haystack.utils import Secret

minimax_chat = OpenAIChatGenerator(
    api_key=Secret.from_env_var("MINIMAX_API_KEY"),
    model="MiniMax-M2.7",
    api_base_url="https://api.minimax.io/v1",
    generation_kwargs={"temperature": 0.7, "max_tokens": 512},
)

messages = [ChatMessage.from_user("What are the Seven Wonders of the Ancient World? List them briefly.")]
result = minimax_chat.run(messages=messages)

print(result["replies"][0].text)

## Build a RAG Pipeline

Now let's build a full Retrieval-Augmented Generation (RAG) pipeline using MiniMax as the LLM.
We'll use:
- **InMemoryDocumentStore** for storing documents
- **SentenceTransformersDocumentEmbedder** for creating document embeddings
- **InMemoryEmbeddingRetriever** for retrieval
- **OpenAIChatGenerator** (pointing to MiniMax) for generation

### Load and Index Documents

We'll use the [Seven Wonders](https://huggingface.co/datasets/bilgeyucel/seven-wonders) dataset.

In [ ]:
from datasets import load_dataset
from haystack import Document

dataset = load_dataset("bilgeyucel/seven-wonders", split="train")
docs = [Document(content=doc["content"], meta=doc["meta"]) for doc in dataset]

print(f"Loaded {len(docs)} documents")

In [ ]:
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

document_store = InMemoryDocumentStore()

doc_embedder = SentenceTransformersDocumentEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])

print(f"Indexed {document_store.count_documents()} documents")

### Define the Prompt Template

Create a prompt that instructs the model to answer based on the retrieved documents.

In [ ]:
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage

system_message = ChatMessage.from_system(
    "You are a helpful assistant that answers questions based on the provided documents. "
    "If the documents don't contain the answer, say so."
)

user_message_template = """\
Answer the question based on these documents:

{% for document in documents %}
Document {{ loop.index }}:
{{ document.content }}
---
{% endfor %}

Question: {{ question }}
Answer:"""

prompt_builder = ChatPromptBuilder(
    template=[system_message, ChatMessage.from_user(user_message_template)]
)

### Assemble the RAG Pipeline

In [ ]:
from haystack import Pipeline
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

text_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
retriever = InMemoryEmbeddingRetriever(document_store)

llm = OpenAIChatGenerator(
    api_key=Secret.from_env_var("MINIMAX_API_KEY"),
    model="MiniMax-M2.7",
    api_base_url="https://api.minimax.io/v1",
    generation_kwargs={"temperature": 0.7, "max_tokens": 1024},
)

rag_pipeline = Pipeline()
rag_pipeline.add_component("text_embedder", text_embedder)
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.add_component("llm", llm)

rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
rag_pipeline.connect("retriever", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder", "llm")

rag_pipeline.draw("rag_pipeline_minimax.png")

### Ask Questions

In [ ]:
question = "Why were people visiting the Temple of Artemis?"

result = rag_pipeline.run(
    data={
        "text_embedder": {"text": question},
        "retriever": {"top_k": 3},
        "prompt_builder": {"question": question},
    }
)

print(result["llm"]["replies"][0].text)

In [ ]:
question = "How did the Colossus of Rhodes collapse?"

result = rag_pipeline.run(
    data={
        "text_embedder": {"text": question},
        "retriever": {"top_k": 3},
        "prompt_builder": {"question": question},
    }
)

print(result["llm"]["replies"][0].text)

## Using MiniMax-M2.7-highspeed

For faster inference, you can switch to the `MiniMax-M2.7-highspeed` model.
It has the same 204K context window but optimized for speed.

In [ ]:
fast_llm = OpenAIChatGenerator(
    api_key=Secret.from_env_var("MINIMAX_API_KEY"),
    model="MiniMax-M2.7-highspeed",
    api_base_url="https://api.minimax.io/v1",
    generation_kwargs={"temperature": 0.7, "max_tokens": 512},
)

messages = [ChatMessage.from_user("Explain what RAG (Retrieval-Augmented Generation) is in 3 sentences.")]
result = fast_llm.run(messages=messages)

print(result["replies"][0].text)

## Summary

In this notebook, you learned how to:
- Use **MiniMax** models with Haystack's `OpenAIChatGenerator` via the OpenAI-compatible API
- Build a complete **RAG pipeline** with MiniMax as the generator
- Switch between **MiniMax-M2.7** (high quality) and **MiniMax-M2.7-highspeed** (optimized for speed)

### Available MiniMax Models

| Model | Context Window | Description |
|-------|---------------|-------------|
| `MiniMax-M2.7` | 204K tokens | Latest flagship model |
| `MiniMax-M2.7-highspeed` | 204K tokens | Speed-optimized variant |

### Key Configuration

To use MiniMax with any Haystack component that supports `OpenAIChatGenerator`:

```python
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.utils import Secret

generator = OpenAIChatGenerator(
    api_key=Secret.from_env_var("MINIMAX_API_KEY"),
    model="MiniMax-M2.7",
    api_base_url="https://api.minimax.io/v1",
)
```

For more details, visit the [MiniMax API documentation](https://platform.minimax.io/).